# 01 — Data Exploration

This notebook loads the sample documents, inspects the raw text, chunks it, and visualises the chunk-length distribution.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

from src.ingestion.loader import DocumentLoader
from src.ingestion.chunker import TextChunker

## 1. Load Documents

In [ ]:
loader = DocumentLoader()
documents = loader.load_directory('../data/sample_docs')

print(f'Documents loaded: {len(documents)}')
for doc in documents:
    meta = doc['metadata']
    print(f"  [{meta['file_type']}] {meta['filename']} — {len(doc['content'])} chars")

## 2. Inspect Raw Content

In [ ]:
for doc in documents:
    print(f"=== {doc['metadata']['filename']} ===")
    print(doc['content'][:500])
    print('...')
    print()

## 3. Chunk the Documents

In [ ]:
chunker = TextChunker(chunk_size=512, chunk_overlap=50)
chunks = chunker.chunk_documents(documents)

print(f'Total chunks: {len(chunks)}')
chunk_lengths = [len(c['content']) for c in chunks]
print(f'Mean length : {np.mean(chunk_lengths):.1f} chars')
print(f'Median length: {np.median(chunk_lengths):.1f} chars')
print(f'Min / Max   : {min(chunk_lengths)} / {max(chunk_lengths)} chars')

## 4. Chunk Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(chunk_lengths, bins=15, color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].axvline(np.mean(chunk_lengths), color='tomato', linestyle='--', label=f'Mean ({np.mean(chunk_lengths):.0f})')
axes[0].axvline(np.median(chunk_lengths), color='orange', linestyle='--', label=f'Median ({np.median(chunk_lengths):.0f})')
axes[0].set_xlabel('Chunk Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chunk Length Histogram')
axes[0].legend()

# Box plot
axes[1].boxplot(chunk_lengths, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_ylabel('Chunk Length (characters)')
axes[1].set_title('Chunk Length Box Plot')
axes[1].set_xticks([])

plt.suptitle('Document Chunk Statistics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sample Chunks

In [ ]:
for i, chunk in enumerate(chunks[:3]):
    print(f'--- Chunk {i} ({len(chunk["content"])} chars) ---')
    print(chunk['content'])
    print(f'Metadata: {chunk["metadata"]}')
    print()